In [3]:
import geopandas as gpd
import pandas as pd

In [2]:
retail_centers = gpd.read_file("geo/postworkshopcentres.gpkg")
retail_centers.columns
# #ignore the retail parks for now
# retail_parks = gpd.read_file("geo/retail_parks_clean.gpkg")
# retail_parks.head()

Index(['RC_ID', 'Total_POI_count', 'H3_cells', 'Retail', 'Dining_and_Drinking',
       'Leisure_Travel', 'Community_Health', 'category_names', 'centre_type',
       'LocalAuthorityNM_1', 'LocalAuthorityNM_2', 'LocalAuthorityNM_3',
       'geometry'],
      dtype='object')

In [3]:
retail_centers.shape

(9622, 13)

In [4]:
ldc = pd.read_csv("data/LDC_Historical_snapshot_2015-01-01-2025-09-30.csv")
ldc_live = ldc[ldc['Status'] == "Live"]
ldc_live.head()

,PremiseId,Id,Name,Businesstype,Unit,Building,CareOf,StreetNo,Street,Postcode,...,Subcategory,MultipleName,HoldingCompany,Latitude,Longitude,Voa,Voa_BusinessRate,CreatedDate,ClosedDate,Status
1,50011505,10011505,Jarndyce Antiquarian Booksellers,Independent,NaN,NaN,NaN,46,Great Russell Street,WC1B 3PA,...,Books - Rare & Secondhand,NaN,NaN,51.51803,-0.12640,224.5,36000.0,1999-03-14 00:00:00.000,NaN,Live
2,50011507,10011507,John Walker Pharmacy,Independent,2-3,Medway Court,NaN,21,Leigh Street,WC1H 9QX,...,Chemists / Toiletries,NaN,NaN,51.52684,-0.12461,0.0,0.0,1999-02-09 00:00:00.000,NaN,Live
3,50011513,10011513,Steven Lee Food And News,Independent,NaN,NaN,NaN,290,Gray's Inn Road,WC1X 8EB,...,Newsagents,NaN,NaN,51.52765,-0.11863,74.0,10000.0,1999-01-17 00:00:00.000,NaN,Live
5,50011521,10011521,Kings Cross Tandoori,Independent,NaN,NaN,NaN,341,Gray's Inn Road,WC1X 8PX,...,Restaurant - Indian,NaN,NaN,51.53051,-0.12228,81.5,16250.0,1999-05-26 00:00:00.000,NaN,Live
8,50011547,10011547,Mabel's Tavern,Multiple,NaN,NaN,NaN,9,Mabledon Place,WC1H 9AZ,...,Public Houses & Inns,Shepherd Neame,Shepherd Neame Limited,51.52784,-0.12748,0.0,0.0,1999-05-09 00:00:00.000,NaN,Live


In [5]:
ldc_live['PremiseId'].nunique()

774361

In [30]:
ldc_live.shape

(827806, 26)

In [6]:
poi_gdf = gpd.GeoDataFrame(
    ldc_live,
    geometry=gpd.points_from_xy(ldc_live.Longitude, ldc_live.Latitude),
    crs="EPSG:4326"
).to_crs(retail_centers.crs)
poi_gdf 

,PremiseId,Id,Name,Businesstype,Unit,Building,CareOf,StreetNo,Street,Postcode,...,MultipleName,HoldingCompany,Latitude,Longitude,Voa,Voa_BusinessRate,CreatedDate,ClosedDate,Status,geometry
1,50011505,10011505,Jarndyce Antiquarian Booksellers,Independent,NaN,NaN,NaN,46,Great Russell Street,WC1B 3PA,...,NaN,NaN,51.518030,-0.126400,224.5,36000.0,1999-03-14 00:00:00.000,NaN,Live,POINT (530093.881 181564.445)
2,50011507,10011507,John Walker Pharmacy,Independent,2-3,Medway Court,NaN,21,Leigh Street,WC1H 9QX,...,NaN,NaN,51.526840,-0.124610,0.0,0.0,1999-02-09 00:00:00.000,NaN,Live,POINT (530192.948 182547.305)
3,50011513,10011513,Steven Lee Food And News,Independent,NaN,NaN,NaN,290,Gray's Inn Road,WC1X 8EB,...,NaN,NaN,51.527650,-0.118630,74.0,10000.0,1999-01-17 00:00:00.000,NaN,Live,POINT (530605.432 182648.024)
5,50011521,10011521,Kings Cross Tandoori,Independent,NaN,NaN,NaN,341,Gray's Inn Road,WC1X 8PX,...,NaN,NaN,51.530510,-0.122280,81.5,16250.0,1999-05-26 00:00:00.000,NaN,Live,POINT (530344.09 182959.555)
8,50011547,10011547,Mabel's Tavern,Multiple,NaN,NaN,NaN,9,Mabledon Place,WC1H 9AZ,...,Shepherd Neame,Shepherd Neame Limited,51.527840,-0.127480,0.0,0.0,1999-05-09 00:00:00.000,NaN,Live,POINT (529991.024 182653.408)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1615095,52976552,14385241,Vacant Property,Dummy,NaN,NaN,NaN,72a,London Road,TN4 0PA,...,NaN,NaN,51.160350,0.257600,0.0,0.0,2021-09-24 00:00:00.000,NaN,Live,POINT (557958.785 142544.845)
1615096,50491901,14385617,Vacant Property,Dummy,25,Spindles Town Square Shopping Centre,NaN,NaN,Market Place,OL1 1JG,...,NaN,NaN,53.541130,-2.114470,0.0,0.0,2008-12-22 00:00:00.000,NaN,Live,POINT (392511.362 404947.189)
1615097,52999920,14385689,Partridge Homes,Independent,NaN,NaN,NaN,120a,Haslucks Green Road,B90 2EH,...,NaN,NaN,52.411440,-1.835070,0.0,0.0,2025-01-09 00:00:00.000,NaN,Live,POINT (411315.312 279282.089)
1615098,52547680,14385785,Vacant Property,Dummy,NaN,NaN,NaN,613,George Street,AB25 3YH,...,NaN,NaN,57.156646,-2.108937,0.0,0.0,2023-08-01 00:00:00.000,NaN,Live,POINT (393509.307 807315.533)


In [7]:
# Spatial Join: Assign each POI the RC_ID of the boundary it falls into
joined = gpd.sjoin(poi_gdf, retail_centers[['RC_ID', 'geometry']], predicate='within')
joined.shape

(533166, 29)

In [8]:
out_gpkg = "output/joined_ldcpoi_with_rcid.gpkg"
layer    = "ldcpoi_rcid_within"

# Write to GeoPackage
joined.to_file(out_gpkg, driver="GPKG", layer=layer)

# Aggregate ldc poi number per retail centre

In [9]:
# count the number of units per Retail Centre
unit_counts = joined.groupby('RC_ID').size().reset_index(name='LDC_retailN')
unit_counts

,RC_ID,LDC_retailN
0,1,1933
1,2,1940
2,3,1456
3,4,988
4,5,1355
...,...,...
9194,9617,1
9195,9618,1
9196,9619,3
9197,9620,3


In [10]:
rc_retailN = retail_centers.merge(unit_counts, on='RC_ID', how='left').fillna(0)
rc_retailN.to_csv("output/retail_centers_with_ldc_retailN.csv", index=False)

In [11]:
rc_retailN.shape

(9622, 14)

### checking the duplicates
- keep the duplicates for now, as for bussiness which has multiple sections, e.g. boots hearing and opticians

In [13]:
dupe_rows = (
    joined[joined.duplicated(subset=["PremiseId"], keep=False)]
    .sort_values("PremiseId")
)
print(f"Duplicate PremiseId rows: {len(dupe_rows)}")
display(dupe_rows.head())

Duplicate PremiseId rows: 43271


,PremiseId,Id,Name,Businesstype,Unit,Building,CareOf,StreetNo,Street,Postcode,...,Latitude,Longitude,Voa,Voa_BusinessRate,CreatedDate,ClosedDate,Status,geometry,index_right,RC_ID
506250,50000531,12990823,Boots Hearing Care,Multiple,NaN,NaN,Boots The Chemist,127a,Kensington High Street,W8 5SF,...,51.500517,-0.192827,NaN,NaN,2014-08-13 00:00:00.000,NaN,Live,POINT (525533.412 179501.028),2580,23
4384,50000531,10000531,Boots Opticians,Multiple,NaN,NaN,Boots The Chemist,127a,Kensington High Street,W8 5SF,...,51.500517,-0.192827,NaN,NaN,2007-05-15 00:00:00.000,NaN,Live,POINT (525533.412 179501.028),2580,23
57295,50000531,10606870,Boots The Chemist,Multiple,NaN,NaN,NaN,127a,Kensington High Street,W8 5SF,...,51.500517,-0.192827,4019.8,1150000.0,2007-05-15 00:00:00.000,NaN,Live,POINT (525533.412 179501.028),2580,23
1062965,50000727,13731601,ParcelShop by Post Office,Multiple,NaN,NaN,Kensington & Chelsea Libraries,12,Phillimore Walk,W8 7RX,...,51.501411,-0.194427,NaN,NaN,2020-05-07 00:00:00.000,NaN,Live,POINT (525419.914 179597.76),2580,23
4431,50000727,10000727,Kensington Central Library,Multiple,NaN,NaN,NaN,12,Phillimore Walk,W8 7RX,...,51.501411,-0.194427,0.0,0.0,1999-01-22 00:00:00.000,NaN,Live,POINT (525419.914 179597.76),2580,23


In [12]:
# Find rows in 'joined' with duplicate PremiseId and the same CreatedDate
dupe_same_created = (
    joined[joined.duplicated(subset=["PremiseId", "CreatedDate"], keep=False)]
    .sort_values(["PremiseId", "CreatedDate"])
)
print(f"Rows with same PremiseId and CreatedDate: {len(dupe_same_created)}")
display(dupe_same_created.head())

Rows with same PremiseId and CreatedDate: 7823


,PremiseId,Id,Name,Businesstype,Unit,Building,CareOf,StreetNo,Street,Postcode,...,Latitude,Longitude,Voa,Voa_BusinessRate,CreatedDate,ClosedDate,Status,geometry,index_right,RC_ID
4384,50000531,10000531,Boots Opticians,Multiple,NaN,NaN,Boots The Chemist,127a,Kensington High Street,W8 5SF,...,51.500517,-0.192827,NaN,NaN,2007-05-15 00:00:00.000,NaN,Live,POINT (525533.412 179501.028),2580,23
57295,50000531,10606870,Boots The Chemist,Multiple,NaN,NaN,NaN,127a,Kensington High Street,W8 5SF,...,51.500517,-0.192827,4019.8,1150000.0,2007-05-15 00:00:00.000,NaN,Live,POINT (525533.412 179501.028),2580,23
751257,50001159,13328771,PC Repair Point,Independent,NaN,NaN,Kahramana,226,Edgware Road,W2 1DH,...,51.518760,-0.168240,NaN,NaN,2017-06-13 00:00:00.000,NaN,Live,POINT (527189.066 181572.14),2296,28
751258,50001159,13328772,Global Choice Travel,Independent,NaN,NaN,Kahramana,226,Edgware Road,W2 1DH,...,51.518760,-0.168240,NaN,NaN,2017-06-13 00:00:00.000,NaN,Live,POINT (527189.066 181572.14),2296,28
744068,50001319,13319488,The Green Man,Multiple,NaN,NaN,NaN,308,Edgware Road,W2 1DY,...,51.520360,-0.170440,0.0,36500.0,2017-05-16 00:00:00.000,NaN,Live,POINT (527031.985 181746.245),2296,28


In [58]:
dupe_rows_latest = dupe_rows.sort_values("CreatedDate", ascending=False).drop_duplicates(subset=["PremiseId"])
print(f"Duplicate PremiseId rows: {len(dupe_rows_latest)}")
dupe_rows_latest

Duplicate PremiseId rows: 17834


,PremiseId,Id,Name,Businesstype,Unit,Building,CareOf,StreetNo,Street,Postcode,...,Latitude,Longitude,Voa,Voa_BusinessRate,CreatedDate,ClosedDate,Status,geometry,index_right,RC_ID
1614227,52606825,14380604,BadRhino,Multiple,G51,Trinity Walk Shopping Centre,Yours,NaN,Marsh Way,WF1 1QU,...,53.68499,-1.49500,NaN,NaN,2025-09-30 00:00:00.000,NaN,Live,POINT (433450.472 421064.72),8421,70
1614281,53061632,14380658,Chicken Shack,Independent,NaN,NaN,The Square Pizza Co,43,Clifton Street,CF24 1LX,...,51.48511,-3.15700,NaN,NaN,2025-09-30 00:00:00.000,NaN,Live,POINT (319759.382 176874.685),5003,1247
1614169,52568366,14380541,The Drunk Tank at Pogue Mahon's,Independent,NaN,NaN,Pogue Mahon's,72,Prince of Wales Road,NR1 1LT,...,52.62916,1.30248,NaN,NaN,2025-09-29 00:00:00.000,NaN,Live,POINT (623594.669 308607.004),6155,386
1614040,53125991,14380406,Specsavers Hearcare,Multiple,4,NaN,Specsavers Opticians,NaN,Pound Lane,NR7 0SR,...,52.63047,1.36301,NaN,NaN,2025-09-29 00:00:00.000,NaN,Live,POINT (627683.067 308942.144),6275,7263
1614025,50907103,14380391,Scrivens Hearing Care,Multiple,NaN,NaN,Scrivens Opticians,209,Plumstead Road,NR1 4AB,...,52.63704,1.32800,NaN,NaN,2025-09-29 00:00:00.000,NaN,Live,POINT (625280.787 309562.655),6147,2922
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3787,50027073,10027073,Seen,Independent,NaN,NaN,NaN,52,Gloucester Road,SW7 4QT,...,51.49790,-0.18392,72.8,24750.0,2000-07-12 00:00:00.000,NaN,Live,POINT (526158.821 179225.361),2596,1618
3328,50024718,10024716,Purcell Room,Independent,1,South Bank Centre,Queen Elizabeth Hall,NaN,South Bank,SE1 8XX,...,51.50550,-0.11731,NaN,NaN,2000-05-02 00:00:00.000,NaN,Live,POINT (530760.387 180187.284),2336,391
3281,50022755,10024524,Eagle Gallery,Independent,NaN,NaN,The Eagle,159,Farringdon Road,EC1R 3AL,...,51.52415,-0.10969,NaN,NaN,2000-04-18 00:00:00.000,NaN,Live,POINT (531235.599 182274.802),2298,21
777,52927707,10014696,Camden Town Library,Independent,NaN,Crowndale Centre,Council Offices,218-220,Eversholt Street,NW1 1BD,...,51.53425,-0.13774,NaN,NaN,1999-12-02 00:00:00.000,NaN,Live,POINT (529261.207 183348.048),7229,219


# Anchor Shop Counts

In [7]:
# Retail centre polygons
retail_centers = gpd.read_file("geo/new_postworkshopcentres05Feb26.gpkg")

# LDC data
ldc = pd.read_csv("data/LDC_Historical_snapshot_2015-01-01-2025-09-30.csv")
ldc_live = ldc[ldc["Status"] == "Live"].copy()

print("Retail centres:", len(retail_centers))
print("Live LDC records:", len(ldc_live))

Retail centres: 9624
Live LDC records: 827806


In [61]:
ldc_live.loc[
    ldc_live["Name"].str.contains("house of fraser", case=False, na=False),
    "Name"
].value_counts()

Name
House of Fraser             12
Highland House Of Fraser     2
Name: count, dtype: int64

In [68]:
import html
import re
import unicodedata
import pandas as pd

# ── 1. YOUR BRAND LIST (trimmed to your spec) ─────────────────────────────────

ANCHOR_BRANDS = {
    "high_street_fashion": [
        "next", "h&m", "zara", "tk maxx", "river island",
        "primark", "jack wills", "white stuff", "fat face",
        "crew clothing", "seasalt", "joules", "lacoste",
        "diesel", "hugo boss", "tommy hilfiger", "ted baker",
        "laura ashley",
    ],
    "department_stores": [
        "john lewis", "harvey nichols", "marks & spencer", "m&s", "ikea","house of fraser"
    ],
    "jewellery_accessories": [
        "h.samuel", "swarovski", "claire's",
    ],
    "sports": [
        "jd sports","sports direct",
    ],
    "health_beauty": [
        "boots",
    ],
    "home_diy": [
        "b&q", "homebase",
    ],
    "leisure_cinema": [
        "odeon", "vue", "cineworld",
    ],
    "electronics_gaming": [
        "apple store", "bose", "game","currys"
    ],
    "fragrance": [
        "the perfume shop",
    ],
}

# ── 2. ALIASES  — based on what's actually in your data ──────────────────────
# "Boots The Chemist" → boots, "McDonald's" style apostrophes, etc.

_ALIASES = {
    "boots the chemist":        "boots",
    "boots opticians":          "boots",
    "boots pharmacy":           "boots",
    "m&s simply food":          "marks & spencer",
    "m & s":                    "marks & spencer",
    "marks and spencer":        "marks & spencer",
    "marks & spencers":         "marks & spencer",
    "h & m":                    "h&m",
    "h and m":                  "h&m",
    "tkmaxx":                   "tk maxx",
    "tk-maxx":                  "tk maxx",
    "jd":                       "jd sports",
    "claire's accessories":     "claire's",
    "claires":                  "claire's",
    "h samuel":                 "h.samuel",
    "h. samuel":                "h.samuel",
    "john lewis & partners":    "john lewis",
    "john lewis and partners":  "john lewis",
    "odeon cinema":             "odeon",
    "odeon cinemas":            "odeon",
    "vue cinema":               "vue",
    "vue cinemas":              "vue",
    "cineworld cinema":         "cineworld",
    "apple":                    "apple store",
    "hugo boss":                "hugo boss",
}

# ── 3. NORMALISATION ──────────────────────────────────────────────────────────

def _normalise(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower().strip()
    text = re.sub(r"[^\w\s&.\'-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ── 4. BUILD LOOKUP (run once) ────────────────────────────────────────────────

def _build_lookup(brand_dict):
    norm_to_cat = {}
    for category, brands in brand_dict.items():
        for b in brands:
            norm_to_cat[_normalise(b)] = category

    alias_to_norm = {}
    for raw_alias, canonical in _ALIASES.items():
        norm_alias = _normalise(raw_alias)
        norm_canonical = _normalise(canonical)
        if norm_canonical in norm_to_cat:
            alias_to_norm[norm_alias] = norm_canonical

    return norm_to_cat, alias_to_norm

NORM_TO_CAT, ALIAS_TO_NORM = _build_lookup(ANCHOR_BRANDS)

# ── 5. MATCH ──────────────────────────────────────────────────────────────────

def _whole_word_match(brand, text):
    """Match brand as whole words only — 'zara' won't match 'zarathustra'."""
    pattern = r"(?<![a-z0-9])" + re.escape(brand) + r"(?![a-z0-9])"
    return bool(re.search(pattern, text))

def _get_category(raw_name, norm_to_cat, alias_to_norm):
    norm = _normalise(raw_name)
    if not norm:
        return None
    if norm in norm_to_cat:        # exact — e.g. "Zara" == "zara"
        return norm_to_cat[norm]
    if norm in alias_to_norm:      # alias — e.g. "Boots The Chemist"
        return norm_to_cat[alias_to_norm[norm]]
    for brand, cat in norm_to_cat.items():   # whole-word substring
        if _whole_word_match(brand, norm):
            return cat
    return None

# ── 6. APPLY ──────────────────────────────────────────────────────────────────

df= ldc_live.copy()  # or your actual DataFrame

df["anchor_category"] = df["Name"].apply(
    lambda n: _get_category(n, NORM_TO_CAT, ALIAS_TO_NORM)
)
df["is_anchor"] = df["anchor_category"].notna()

# ── Count-based filter - manual check ────────────────────────────────────────────────────────
# df[df["is_anchor"]]["Name"].value_counts().rename_axis("Name").reset_index(name="count").to_csv(
#     "output/anchor_name_counts.csv", index=False
# )
# Brands that should be kept regardless of count (known low-footprint anchors)
COUNT_EXEMPT = {
    "M&S Opticians",
    "M&S Food To Go", 
    "Harvey Nichols",
    "Hugo Boss",
    "Laura Ashley Home",
    "Zara Home",
    "H&M Kids",
    "Laura Ashley"

}

# Count how many times each name appears across the full dataset
name_counts = df["Name"].map(df["Name"].value_counts())

# Cancel the flag if count < 10 AND not in the exempt list
low_count_mask = (name_counts <= 10) & (~df["Name"].isin(COUNT_EXEMPT))
df.loc[low_count_mask, "is_anchor"] = False
df.loc[low_count_mask, "anchor_category"] = None

df[df["is_anchor"]]["Name"].value_counts().head()

Name
Boots The Chemist    1845
Boots Opticians       535
Sports Direct         463
M&S Simply Food       406
JD Sports             400
Name: count, dtype: int64

# mannual check for threshold to upgrade the discric centre to town centre

In [6]:
df = pd.read_csv('output/anchor_name_classified_thr3.csv')
df.columns

Index(['fid', 'RC_ID', 'Total_POI_count', 'H3_cells', 'Retail',
       'Dining_and_Drinking', 'Leisure_Travel', 'Community_Health',
       'category_names', 'centre_type', 'LocalAuthorityNM_1',
       'LocalAuthorityNM_2', 'LocalAuthorityNM_3', 'was_split',
       'Original_POI_count', 'Parent_ID', 'unit_counts', 'anchor_counts',
       'anchor_shops', 'LDC_retailN', 'LAD24NM', 'CountyName', 'RegionName',
       'gsscode', 'tr_retailN', 'group_id', 'group_LDC_retailN',
       'group_RGN_rank', 'group_CTY_rank', 'group_LAD_rank', 'group_BUA_rank',
       'Classification'],
      dtype='object')

In [14]:
df.loc[
    (df["Classification"] == "District Centre") & 
    (df["anchor_counts"]==2),
    "RC_ID"
]

84       319
155      990
901      964
1525     384
1811     559
2093     831
2110     173
2146     254
2413     405
2452     570
2510     510
2586      51
2896     679
3213     304
3392     635
3428     478
4132     256
4384     549
4596     429
5142     531
5181     697
5276     535
5284     488
5605    1092
6175     987
6224     661
6891     440
6920     618
7167     711
7337     595
7353     775
7493     609
7905     313
7951     766
8059     914
8290     451
Name: RC_ID, dtype: int64

In [72]:
df.loc[
    df["Classification"] == "Town Centre",
    "anchor_counts"
].value_counts()

anchor_counts
3     32
1     23
2     17
0     16
4     16
7     14
9     13
5     11
8     10
12     9
10     9
6      9
13     8
17     6
15     4
14     3
11     2
20     2
16     2
24     1
Name: count, dtype: int64